In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t082'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 2026-04-13 06:04:35 and 2026-04-13 06:09:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/ListProducts` of type `apm.operation` in the `apm` domain. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', {'start': '2026-04-13T06:04:35+08:00', 'end': '2026-04-13T06:09:32+08:00'}, {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">product-catalog 流量下跌告警<alert_event event_id="95ffe3db6d2c4bd170d85e0e42689865" trans_id="uf83tub3e93a997ecte35l2g54" rule_id="e876641e-1e5e-49d4-9e38-9233255b5baf" rule_name="product-catalog 流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their corresponding entity sets
kpi_entity_sets = metrics_data[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Key entity sets include `k8s.node`, `apm.service.legacy`, `apm.operation`, and `apm.instance`. Relevant KPIs for anomaly detection include metrics like `node_memory_usage_rate`, `node_disk_usage_rate`, `error_rate`, `slow`, and `workload`. For a comprehensive view of all KPIs, the full table should be examined.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0           node_ready_status            k8s.node
2010   node_memory_usage_rate            k8s.node
3886     node_disk_usage_rate            k8s.node
5226    node_not_ready_status            k8s.node
7772   node_pod_running_count            k8s.node
...                       ...                 ...
75025                workload  apm.service.legacy
75037                    slow  apm.service.legacy
75052                   error  apm.service.legacy
84268              error_rate       apm.operation
84270              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert 'time' column to numeric and calculate global P95 thresholds
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Calculate global P95 thresholds for each 'component-KPI' time series
global_p95_thresholds = metrics_data.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
global_p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

# Filter data within the given time window
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.strptime(alert_window['start'], "%Y-%m-%dT%H:%M:%S%z").astimezone(timezone).timestamp() * 1_000_000)
end_time = int(datetime.strptime(alert_window['end'], "%Y-%m-%dT%H:%M:%S%z").astimezone(timezone).timestamp() * 1_000_000)

filtered_metrics = metrics_data[(metrics_data['time'] >= start_time) & (metrics_data['time'] <= end_time)]

global_p95_thresholds, filtered_metrics

```
Out[3]:
```
The global P95 thresholds were calculated for 270 unique 'component-KPI' time series in the `metrics.parquet` file. After filtering the data to the specified time window (2026-04-13 06:04:35 to 2026-04-13 06:09:32 UTC+8), 6,274 relevant metric records were identified. These filtered metrics can now be analyzed further to detect anomalies by comparing their values against the precomputed P95 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

(                                      entity_name                               metric  p95_threshold
0                                                     apiserver_concurrent_connections       6.000000
1                                                   apiserver_non_2xx_read_request_qps       0.198066
2                                                  apiserver_non_2xx_write_request_qps       0.053647
3                                                        apiserver_requests_per_second      21.224827
4                                                            cluster_failed_node_count       0.000000
..                                            ...                                  ...            ...
265                                      shipping                             workload    7534.950000
266   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds       0.002381
267   shipping::oteldemo.ShippingService/GetQuote                        request_count    3767.300000
268  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds       0.000012
269  shipping::oteldemo.ShippingService/ShipOrder                        request_count    3767.650000

[270 rows x 3 columns],                    time domain          entity_set entity_id     entity_name             metric         value                                     metric_set_id service
120    1776031475000000    k8s            k8s.node                            node_ready_status  1.000000e+00  k8s@metric_set@k8s.metric.high_level_metric_node    None
121    1776031505000000    k8s            k8s.node                            node_ready_status  1.000000e+00  k8s@metric_set@k8s.metric.high_level_metric_node    None
122    1776031535000000    k8s            k8s.node                            node_ready_status  1.000000e+00  k8s@metric_set@k8s.metric.high_level_metric_node    None
123    1776031565000000    k8s            k8s.node                            node_ready_status  1.000000e+00  k8s@metric_set@k8s.metric.high_level_metric_node    None
124    1776031595000000    k8s            k8s.node                            node_ready_status  1.000000e+00  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...                 ...       ...             ...                ...           ...                                               ...     ...
83987  1776031745000000    apm  apm.service.legacy            recommendation           workload  2.230000e+02                                              None    None
83988  1776031745000000    apm  apm.service.legacy                  shipping                cpu  4.231115e-03                                              None    None
83989  1776031745000000    apm  apm.service.legacy                  shipping            latency  1.151554e-03                                              None    None
83990  1776031745000000    apm  apm.service.legacy                  shipping                mem  6.735189e+06                                              None    None
83991  1776031745000000    apm  apm.service.legacy                  shipping           workload  4.472000e+03                                              None    None

[6274 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered metrics with the global P95 thresholds
merged_data = filtered_metrics.merge(global_p95_thresholds, on=['entity_name', 'metric'], how='left')

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

# Group anomalies by component and KPI for better understanding
anomalies_grouped = anomalies.groupby(['entity_name', 'metric']).size().reset_index(name='anomaly_count')

anomalies, anomalies_grouped

```
Out[4]:
```
During the specified time window (2026-04-13 06:04:35 to 2026-04-13 06:09:32 UTC+8), 239 metric anomalies were detected across various components and KPIs. These anomalies include metrics such as `node_memory_usage_rate`, `deployment_cpu_usage_total`, `deployment_memory_usage_total`, and `avg_request_latency_seconds`. A total of 68 unique 'component-KPI' pairs exhibited anomalies, with some components like `product-catalog` and `recommendation` showing latency-related issues. These findings suggest potential performance or resource-related problems in the system.

The original code execution output of IPython Kernel is also provided below for reference:

(                  time domain          entity_set entity_id entity_name                  metric         value                                     metric_set_id service  p95_threshold
220   1776031475000000    k8s            k8s.node                        node_memory_usage_rate  9.046449e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   9.023034e+01
221   1776031505000000    k8s            k8s.node                        node_memory_usage_rate  9.046318e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   9.023034e+01
222   1776031535000000    k8s            k8s.node                        node_memory_usage_rate  9.046491e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   9.023034e+01
223   1776031565000000    k8s            k8s.node                        node_memory_usage_rate  9.048549e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   9.023034e+01
224   1776031595000000    k8s            k8s.node                        node_memory_usage_rate  9.049839e+01  k8s@metric_set@k8s.metric.high_level_metric_node    None   9.023034e+01
...                ...    ...                 ...       ...         ...                     ...           ...                                               ...     ...            ...
6085  1776031685000000    apm  apm.service.legacy              currency                 latency  1.682446e-03                                              None    None   1.672456e-03
6089  1776031685000000    apm  apm.service.legacy                 email                 latency  2.931080e-03                                              None    None   2.845862e-03
6154  1776031715000000    apm  apm.service.legacy              currency                 latency  1.703316e-03                                              None    None   1.672456e-03
6223  1776031745000000    apm  apm.service.legacy              currency                 latency  1.679314e-03                                              None    None   1.672456e-03
6249  1776031745000000    apm  apm.service.legacy             inventory                     mem  6.392245e+08                                              None    None   6.392068e+08

[239 rows x 10 columns],                                           entity_name                               metric  anomaly_count
0                                                      apiserver_non_2xx_write_request_qps              1
1                                                               deployment_cpu_usage_total             17
2                                                           deployment_cpu_usage_vs_limits             16
3                                                         deployment_cpu_usage_vs_requests             11
4                                                            deployment_memory_usage_total             22
..                                                ...                                  ...            ...
63  product-catalog::oteldemo.ProductCatalogServic...          avg_request_latency_seconds              2
64  product-catalog::oteldemo.ProductCatalogServic...          avg_request_latency_seconds              2
65                                     recommendation                              latency              1
66                    recommendation-75b9f75b58-7s4sd          avg_request_latency_seconds              1
67  recommendation::/oteldemo.RecommendationServic...          avg_request_latency_seconds              1

[68 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the specified time window
start_time_ns = start_time * 1_000  # Convert microseconds to nanoseconds
end_time_ns = end_time * 1_000  # Convert microseconds to nanoseconds
filtered_traces = traces_data[(traces_data['startTime'] >= start_time_ns) & (traces_data['endTime'] <= end_time_ns)]

# Identify traces involving the 'product-catalog' service
product_catalog_traces = filtered_traces[filtered_traces['serviceName'].str.contains('product-catalog', case=False, na=False)]

# Analyze the call chain to determine the most downstream faulty service
# A downstream service is identified by its absence as a 'parentSpanId' in the 'spanId' column
downstream_faulty_services = product_catalog_traces[~product_catalog_traces['parentSpanId'].isin(product_catalog_traces['spanId'])]

product_catalog_traces, downstream_faulty_services

```
Out[5]:
```
During the specified time window (2026-04-13 06:04:35 to 2026-04-13 06:09:32 UTC+8), traces involving the `product-catalog` service were successfully filtered. These traces include operations such as `oteldemo.ProductCatalogService/GetProduct` and `oteldemo.ProductCatalogService/ListProducts`. The analysis identified the most downstream faulty spans, which are spans without a parent in the call chain. These spans indicate potential issues originating from the `product-catalog` service itself or its interactions with other services. Further investigation into these spans is required to pinpoint the exact root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                                     spanName  ...  __topic__      __tag__:__pack_id__                         hostname                                             events links
191994  9ffa2599e99b9c402b130321b13b1f88  56b5267ef07e6dff  084c98477c400448    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191995  1c409079e60aca90b44de5c47dbbec56  64d4feae585f4a0d  1df96ad383e831c4    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191996  af6a946f0337e2d0f898be064f1a9eee  163890665f3d0d75  aab13947e94f72f4    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191997  3cdcd5e1ea692da550e7e77d1cdc0087  a4c7abb69e6bf5bf  6d2242d87fc4039e    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191998  d5ab74c578dcd314074666ca579f4b66  34ec3d597bae4732  a5ab46f401b194de    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
...                                  ...               ...               ...  ...                                          ...  ...        ...                      ...                              ...                                                ...   ...
288377  bb5d7b2209b97766937889d61ed1df46  84aa967467dbbfe7  8a5d62bc2c3c57af    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288378  691a28285f7b9bca7f187b804dc1b505  cd4b08f9eb120213  87ba0cdf83277507    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288379  9b9a3b5da117dcec5366dd683fbf1094  be1e5efa7bbb2986  189d601f42e49cd9    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288380  024550daaa391532650978f7950b4152  894b1bd49d0a2053  8e1696d869041be2    2  oteldemo.ProductCatalogService/ListProducts  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q                                               None  None
288381  bb5d7b2209b97766937889d61ed1df46  79abbe7913ea5514  2ef26f935c61e837    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None

[6245 rows x 21 columns],                                  traceId            spanId      parentSpanId kind                                     spanName  ...  __topic__      __tag__:__pack_id__                         hostname                                             events links
191994  9ffa2599e99b9c402b130321b13b1f88  56b5267ef07e6dff  084c98477c400448    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191995  1c409079e60aca90b44de5c47dbbec56  64d4feae585f4a0d  1df96ad383e831c4    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191996  af6a946f0337e2d0f898be064f1a9eee  163890665f3d0d75  aab13947e94f72f4    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191997  3cdcd5e1ea692da550e7e77d1cdc0087  a4c7abb69e6bf5bf  6d2242d87fc4039e    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
191998  d5ab74c578dcd314074666ca579f4b66  34ec3d597bae4732  a5ab46f401b194de    2    oteldemo.ProductCatalogService/GetProduct  ...             46FA981222351EE3-5856ED  product-catalog-79f7b8c4b-rmpgh  [{"attributes":{},"name":"Product Found","time...  None
...                                  ...               ...               ...  ...                                          ...  ...        ...                      ...                              ...                                                ...   ...
288377  bb5d7b2209b97766937889d61ed1df46  84aa967467dbbfe7  8a5d62bc2c3c57af    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288378  691a28285f7b9bca7f187b804dc1b505  cd4b08f9eb120213  87ba0cdf83277507    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288379  9b9a3b5da117dcec5366dd683fbf1094  be1e5efa7bbb2986  189d601f42e49cd9    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None
288380  024550daaa391532650978f7950b4152  894b1bd49d0a2053  8e1696d869041be2    2  oteldemo.ProductCatalogService/ListProducts  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q                                               None  None
288381  bb5d7b2209b97766937889d61ed1df46  79abbe7913ea5514  2ef26f935c61e837    2    oteldemo.ProductCatalogService/GetProduct  ...             353AC8307CF7248F-58322F  product-catalog-79f7b8c4b-lmp9q  [{"attributes":{},"name":"Product Found","time...  None

[6245 rows x 21 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert '_time_' column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs within the specified time window
filtered_logs = logs_data[(logs_data['_time_'] >= start_time) & (logs_data['_time_'] <= end_time)]

# Filter logs related to the 'product-catalog' service
product_catalog_logs = filtered_logs[filtered_logs['_container_name_'].str.contains('product-catalog', case=False, na=False)]

# Look for error messages or anomalies in the logs
error_logs = product_catalog_logs[product_catalog_logs['content'].str.contains('error', case=False, na=False)]

product_catalog_logs, error_logs

```
Out[6]:
```
No logs related to the `product-catalog` service were found within the specified time window (2026-04-13 06:04:35 to 2026-04-13 06:09:32 UTC+8). Additionally, no error messages or anomalies were detected in the logs for this service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns], Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns])```
```